# ViT Video Food Classifier -- Google Colab Notebook

Fine-grained 16-class food classification using MobileViT-XXS / ViT-B + BiLSTM.

**Run order:** clone -> mount drive -> install -> env check -> data download -> upload to HF -> train -> evaluate -> export -> upload model

## 1. Setup -- Clone & Install

In [ ]:
import os, sys, subprocess

REPO_URL = "https://github.com/whispr-messenger/moderation-service.git"
REPO_DIR = "/content/moderation-service"
BRANCH = "WHISPR-668"  # change to "main" if your branch is not pushed yet

if not os.path.exists(REPO_DIR):
    result = subprocess.run(
        ["git", "clone", "-b", BRANCH, REPO_URL, REPO_DIR],
        capture_output=True, text=True,
    )
    if result.returncode != 0:
        print(f"Git clone failed:
{result.stderr}")
        print(f"
Falling back to main branch...")
        result = subprocess.run(
            ["git", "clone", REPO_URL, REPO_DIR],
            capture_output=True, text=True,
        )
        if result.returncode != 0:
            raise RuntimeError(f"Clone failed: {result.stderr}")
        print("Cloned main branch successfully")
    else:
        print(f"Cloned {BRANCH} successfully")
else:
    print(f"Repository already cloned at {REPO_DIR}")

VIT_DIR = os.path.join(REPO_DIR, "src", "vit_video")
SRC_DIR = os.path.join(REPO_DIR, "src")
os.chdir(VIT_DIR)
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

print(f"Working directory: {os.getcwd()}")
print(f"sys.path includes: {SRC_DIR}")

In [ ]:
# Mount Google Drive (optional - for checkpoint persistence)
# Skip this cell if you don't want Drive sync.
DRIVE_CHECKPOINT_DIR = None
DRIVE_CHECKPOINT_PATH = None

try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_CHECKPOINT_DIR = '/content/drive/MyDrive/whispr-checkpoints'
    os.makedirs(DRIVE_CHECKPOINT_DIR, exist_ok=True)
    DRIVE_CHECKPOINT_PATH = os.path.join(DRIVE_CHECKPOINT_DIR, 'best_food_classifier.pth')
    print(f'Drive checkpoint dir: {DRIVE_CHECKPOINT_DIR}')
    if os.path.exists(DRIVE_CHECKPOINT_PATH):
        print(f'Found existing checkpoint: {DRIVE_CHECKPOINT_PATH}')
    else:
        print('No existing checkpoint -- training will start from scratch')
except Exception as e:
    print(f'Drive not mounted ({e}) -- training will work but no checkpoint sync')

In [ ]:
# Install dependencies
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'icrawler', 'huggingface_hub'], check=True)
print('Dependencies installed')

## 2. Environment Check

In [ ]:
import importlib
from pathlib import Path

print(f'Python {sys.version}')
print()
for pkg in ['torch', 'torchvision', 'timm', 'cv2', 'sklearn', 'icrawler', 'huggingface_hub']:
    try:
        m = importlib.import_module(pkg)
        print(f'  {pkg:18s} {getattr(m, "__version__", "ok")}')
    except ImportError:
        print(f'  {pkg:18s} NOT INSTALLED')

import torch
print(f'\nCUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

from vit_video.utils.hardware import get_device, print_device_info
device = get_device()
print_device_info(hint_cuda=True)

## 3. HuggingFace Authentication

Add `HF_TOKEN` to **Colab Secrets** (key icon in the left sidebar) for upload/private dataset access.
Public dataset download works without a token.

In [ ]:
HF_TOKEN = ''

# Try Colab secrets first
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN') or ''
    if HF_TOKEN:
        print('HF token loaded from Colab secrets')
except Exception:
    pass

# Fallback: env var or .env file
if not HF_TOKEN:
    HF_TOKEN = os.environ.get('HF_TOKEN', '')
    env_path = Path(REPO_DIR) / '.env'
    if not HF_TOKEN and env_path.exists():
        for line in env_path.read_text().splitlines():
            if line.startswith('HF_TOKEN='):
                HF_TOKEN = line.split('=', 1)[1].strip().strip('"').strip("'")

if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN
    print('HF_TOKEN set')
else:
    print('No HF_TOKEN -- public download still works, but uploads will be skipped')

## 4. Data -- Categories & Paths

In [ ]:
from vit_video.paths import DEFAULT_DATASET_DIR, DEFAULT_FRAMES_DIR, PACKAGE_ROOT
from vit_video.data.splits import frames_directory_has_images

DATASET_DIR = DEFAULT_DATASET_DIR
FRAMES_DIR = DEFAULT_FRAMES_DIR
FRAMES_DIR.mkdir(parents=True, exist_ok=True)

HEALTH_LABELS = {
    'fruits': 'healthy', 'vegetables': 'healthy', 'salads': 'healthy',
    'seafood': 'healthy', 'grilled_meat': 'healthy', 'grain_bowls': 'healthy',
    'soups': 'healthy', 'smoothies': 'healthy',
    'burgers': 'unhealthy', 'pizza': 'unhealthy', 'fried_food': 'unhealthy',
    'desserts': 'unhealthy', 'candy_sweets': 'unhealthy',
    'salty_snacks': 'unhealthy', 'sugary_drinks': 'unhealthy',
    'not_food': 'not_food',
}

print(f'Dataset dir: {DATASET_DIR}')
print(f'Frames dir:  {FRAMES_DIR}')
print(f'Classes:     {len(HEALTH_LABELS)} ({sum(1 for v in HEALTH_LABELS.values() if v == "healthy")} healthy + '
      f'{sum(1 for v in HEALTH_LABELS.values() if v == "unhealthy")} unhealthy + 1 not_food)')

## 5. Dataset Download

**Option A:** Download from HuggingFace (fast, ~30s).  
**Option B:** Scrape from Bing if HF download fails or repo is empty (~30 min).

In [ ]:
HF_DATASET_REPO = 'maia2000/food-classifier-dataset'

# Try HuggingFace first
downloaded_from_hf = False
try:
    from huggingface_hub import snapshot_download
    print(f'Trying to download from HuggingFace ({HF_DATASET_REPO})...')
    snapshot_download(
        repo_id=HF_DATASET_REPO,
        repo_type='dataset',
        local_dir=str(DATASET_DIR),
        allow_patterns=['frames/**', 'video_split_manifest.json'],
        token=HF_TOKEN if HF_TOKEN else None,
    )
    if frames_directory_has_images(FRAMES_DIR):
        downloaded_from_hf = True
        print(f'Dataset downloaded from HuggingFace')
    else:
        print('HF download succeeded but no frames found -- will scrape')
except Exception as e:
    print(f'HF download failed: {e}')
    print('Will scrape from Bing instead...')

print(f'\nDownloaded from HF: {downloaded_from_hf}')
print(f'Frames present: {frames_directory_has_images(FRAMES_DIR)}')

In [ ]:
# Scrape from Bing if HF download failed (or to top up missing classes)
# Already-downloaded keywords are skipped automatically.

FORCE_SCRAPE = False  # Set True to scrape even if HF download succeeded

if not downloaded_from_hf or FORCE_SCRAPE:
    import re, shutil, logging
    from icrawler.builtin import BingImageCrawler

    IMAGES_PER_KEYWORD = 20
    NON_FOOD_CLASSES = {'not_food'}

    # 16-class keyword dictionary (~27 keywords/class)
    FOOD_CATEGORIES = {
        'fruits': ['banana fruit', 'apple fruit', 'orange citrus', 'mixed berries', 'mango sliced',
                   'watermelon slice', 'kiwi halved', 'pineapple sliced', 'grapes bunch', 'pomegranate',
                   'papaya sliced', 'blueberries', 'strawberries', 'peach sliced', 'pear green',
                   'cherry bowl', 'plum purple', 'cantaloupe', 'passion fruit', 'dragon fruit',
                   'fig fresh', 'apricot', 'raspberry', 'tangerine', 'coconut', 'grapefruit', 'avocado'],
        'vegetables': ['broccoli steamed', 'roasted vegetables', 'vegetable stir fry', 'grilled zucchini',
                       'sauteed mushrooms', 'asparagus grilled', 'bell pepper sliced', 'cauliflower roasted',
                       'brussels sprouts', 'spinach sauteed', 'green beans', 'sweet corn cob', 'eggplant grilled',
                       'artichoke cooked', 'beets roasted', 'carrots roasted', 'cabbage sauteed', 'kale cooked',
                       'peas green', 'radish salad', 'turnip roasted', 'celery sticks', 'bok choy',
                       'snap peas', 'okra cooked', 'leek sliced', 'pumpkin roasted'],
        'salads': ['green salad bowl', 'caesar salad', 'mixed greens salad', 'greek salad feta',
                   'caprese salad', 'quinoa salad', 'cobb salad', 'arugula salad', 'tabbouleh',
                   'pasta salad', 'coleslaw', 'beet goat cheese salad', 'chickpea salad', 'nicoise salad',
                   'fruit salad', 'potato salad', 'southwest salad', 'thai mango salad', 'seaweed salad',
                   'fattoush salad', 'spinach strawberry salad', 'broccoli salad', 'lentil salad warm',
                   'asian sesame salad', 'waldorf salad', 'kale spinach salad', 'cucumber tomato salad'],
        'seafood': ['grilled salmon', 'sushi roll', 'baked cod', 'grilled shrimp', 'poke bowl tuna',
                    'sashimi platter', 'fish tacos', 'steamed mussels', 'lobster tail grilled', 'scallops seared',
                    'grilled tuna steak', 'shrimp cocktail', 'crab legs steamed', 'oysters fresh', 'calamari grilled',
                    'tilapia baked', 'ceviche fresh', 'smoked salmon bagel', 'sardines grilled', 'sea bass baked',
                    'trout pan seared', 'clam chowder', 'fish curry', 'prawn stir fry', 'mackerel grilled',
                    'octopus grilled', 'anchovy plate'],
        'grilled_meat': ['grilled chicken breast', 'turkey breast sliced', 'hard boiled eggs', 'grilled steak',
                         'lamb chops grilled', 'pork tenderloin roasted', 'chicken kebab', 'beef stir fry',
                         'rotisserie chicken', 'turkey meatballs', 'chicken thighs grilled', 'sirloin steak',
                         'venison steak', 'duck breast seared', 'beef brisket smoked', 'baked chicken wings',
                         'pork chops grilled', 'flank steak', 'rabbit roasted', 'quail grilled',
                         'beef jerky', 'chicken satay', 'shawarma chicken', 'pulled pork',
                         'filet mignon', 'rack of lamb', 'beef tartare'],
        'grain_bowls': ['quinoa bowl', 'brown rice bowl', 'chickpea buddha bowl', 'overnight oats',
                        'oatmeal breakfast', 'whole grain toast avocado', 'granola yogurt', 'couscous bowl',
                        'farro grain bowl', 'barley soup', 'acai bowl', 'chia pudding',
                        'rice cakes', 'cottage cheese bowl', 'tofu rice bowl', 'black bean rice bowl',
                        'burrito bowl', 'bibimbap', 'risotto mushroom', 'congee', 'polenta creamy',
                        'muesli bowl', 'bulgur wheat bowl', 'wild rice pilaf', 'tabbouleh grain', 'hummus bowl',
                        'poke rice bowl'],
        'soups': ['lentil soup', 'miso soup', 'vegetable soup', 'minestrone', 'tomato soup',
                  'chicken noodle soup', 'butternut squash soup', 'gazpacho', 'pho noodle soup', 'wonton soup',
                  'corn chowder', 'mushroom soup', 'black bean soup', 'split pea soup', 'french onion soup',
                  'egg drop soup', 'coconut curry soup', 'borscht', 'hot and sour soup', 'ramen soup',
                  'tortilla soup', 'broccoli cheddar soup', 'lobster bisque', 'pozole', 'laksa',
                  'udon soup', 'bone broth'],
        'smoothies': ['green smoothie', 'berry smoothie', 'mango smoothie', 'banana smoothie',
                      'protein smoothie', 'acai smoothie', 'spinach smoothie', 'strawberry banana smoothie',
                      'peanut butter smoothie', 'avocado smoothie', 'coconut smoothie', 'blueberry smoothie',
                      'oat milk smoothie', 'detox green juice', 'carrot orange juice', 'beet juice',
                      'watermelon juice', 'celery juice', 'ginger turmeric shot', 'matcha latte',
                      'kombucha', 'fresh orange juice', 'cucumber mint water', 'lemon water',
                      'iced green tea', 'coconut water', 'almond milk smoothie'],
        'burgers': ['cheeseburger', 'bacon cheeseburger', 'double cheeseburger', 'big mac', 'slider burger',
                    'mushroom swiss burger', 'bbq bacon burger', 'wagyu burger', 'turkey burger',
                    'veggie burger', 'patty melt', 'western burger', 'jalapeno burger', 'blue cheese burger',
                    'smash burger', 'whopper burger', 'breakfast burger', 'truffle burger', 'chipotle burger',
                    'teriyaki burger', 'elk burger', 'bison burger', 'lamb burger', 'hot dog bun',
                    'gourmet burger', 'fast food burger', 'in n out burger'],
        'pizza': ['pepperoni pizza', 'margherita pizza', 'meat lovers pizza', 'hawaiian pizza',
                  'deep dish pizza', 'thin crust pizza', 'stuffed crust pizza', 'buffalo chicken pizza',
                  'bbq chicken pizza', 'white pizza', 'supreme pizza', 'four cheese pizza',
                  'mushroom pizza', 'veggie pizza', 'calzone', 'pizza rolls', 'flatbread pizza',
                  'sicilian pizza', 'detroit pizza', 'neapolitan pizza', 'french bread pizza',
                  'bagel bites', 'pizza hut box', 'dominos pizza', 'frozen pizza', 'garlic knots',
                  'pizza dough'],
        'fried_food': ['french fries', 'fried chicken bucket', 'chicken wings deep fried', 'onion rings',
                       'mozzarella sticks', 'corn dog', 'fish and chips', 'chicken nuggets',
                       'fried calamari', 'tempura shrimp', 'jalapeno poppers', 'fried oreos',
                       'fried pickles', 'fried cheese curds', 'chicken tenders', 'fried wontons',
                       'hush puppies', 'fried zucchini', 'corn fritters', 'fried shrimp basket',
                       'tater tots', 'fried ravioli', 'poutine', 'loaded fries', 'curly fries',
                       'waffle fries', 'sweet potato fries'],
        'desserts': ['chocolate cake', 'glazed donut', 'cupcake frosting', 'cheesecake slice',
                     'churros', 'cinnamon roll', 'brownie chocolate', 'pancakes syrup',
                     'funnel cake', 'cream puff', 'tiramisu', 'pie slice cream',
                     'waffle chocolate', 'eclair pastry', 'croissant pastry', 'baklava',
                     'apple pie', 'red velvet cake', 'birthday cake', 'creme brulee',
                     'panna cotta', 'profiteroles', 'danish pastry', 'scone clotted cream',
                     'tres leches cake', 'bread pudding', 'glazed donuts box'],
        'candy_sweets': ['candy unwrapping', 'chocolate bar', 'gummy bears', 'jelly beans',
                         'cotton candy', 'caramel popcorn', 'lollipop', 'ice cream sundae',
                         'ice cream cone', 'frozen yogurt', 'popsicle', 'milkshake thick',
                         'chocolate truffles', 'fudge squares', 'marshmallow smores', 'toffee candy',
                         'taffy pulled', 'rock candy', 'candy corn', 'licorice candy',
                         'chocolate mousse', 'm and m candy', 'skittles candy', 'kit kat',
                         'snickers bar', 'reeses peanut butter cup', 'gummy worms'],
        'salty_snacks': ['potato chips', 'nachos cheese', 'cheese puffs', 'pretzel bites',
                         'corn chips salsa', 'popcorn buttered', 'trail mix', 'crackers cheese',
                         'doritos chips', 'cheetos bag', 'pringles can', 'tortilla chips',
                         'goldfish crackers', 'chex mix', 'beef jerky snack', 'pork rinds',
                         'sunflower seeds', 'mixed nuts salted', 'rice crackers', 'cheese crackers',
                         'onion dip chips', 'hummus chips', 'breadsticks', 'cheese balls',
                         'bugles cone', 'funyuns', 'combos pretzel'],
        'sugary_drinks': ['cola pouring glass', 'milkshake straw', 'energy drink can', 'iced frappuccino',
                          'bubble tea boba', 'slurpee frozen', 'lemonade pitcher', 'sweet iced tea',
                          'hot chocolate whipped cream', 'mocha sweet cream', 'fruit punch bowl', 'capri sun',
                          'kool aid pitcher', 'mountain dew soda', 'sprite soda', 'fanta orange',
                          'root beer float', 'cream soda', 'dr pepper soda', 'gatorade bottle',
                          'arizona iced tea', 'starbucks frappuccino', 'chocolate milk', 'strawberry milkshake',
                          'vanilla milkshake', 'mango lassi', 'pina colada'],
        'not_food': ['cat sleeping couch', 'dog playing park', 'bird flying sky', 'horse field',
                     'fish aquarium', 'rabbit pet', 'person walking street', 'people jogging',
                     'crowd concert', 'children playground', 'office workers meeting', 'students classroom',
                     'car driving highway', 'bicycle path', 'airplane sky', 'train station',
                     'sunset ocean beach', 'mountain hiking', 'flowers garden', 'snow forest',
                     'waterfall tropical', 'desert sand dunes', 'city skyline night', 'construction crane',
                     'shopping mall', 'bridge architecture', 'book library', 'laptop desk work',
                     'guitar music', 'painting art museum', 'phone texting', 'camera lens',
                     'soccer ball field', 'basketball court', 'swimming pool', 'yoga mat',
                     'empty plate', 'kitchen counter', 'cutting board', 'cooking pot',
                     'grocery aisle', 'restaurant empty tables', 'menu card', 'vending machine',
                     'kitchen utensils', 'oven appliance', 'refrigerator open', 'coffee machine',
                     'wine glasses empty', 'barbeque grill empty', 'dining table no food'],
    }

    def download_images(frames_dir, categories, images_per_keyword=20):
        for category, keywords in categories.items():
            cat_dir = frames_dir / category
            cat_dir.mkdir(parents=True, exist_ok=True)
            for keyword in keywords:
                slug = re.sub(r'[^a-z0-9]+', '_', keyword.lower()).strip('_')
                if len(list(cat_dir.glob(f'{slug}_frame_*.jpg'))) >= images_per_keyword:
                    continue
                tmp_dir = frames_dir / '_tmp_download'
                if tmp_dir.exists():
                    shutil.rmtree(tmp_dir)
                tmp_dir.mkdir()
                try:
                    query = keyword if category in NON_FOOD_CLASSES else keyword + ' food'
                    crawler = BingImageCrawler(storage={'root_dir': str(tmp_dir)}, log_level=logging.ERROR)
                    crawler.crawl(keyword=query, max_num=images_per_keyword)
                    idx = 0
                    for img_file in sorted(tmp_dir.iterdir()):
                        if img_file.suffix.lower() in ('.jpg', '.jpeg', '.png', '.webp'):
                            shutil.move(str(img_file), str(cat_dir / f'{slug}_frame_{idx:04d}.jpg'))
                            idx += 1
                except Exception as e:
                    print(f'  [WARN] {category}/{keyword}: {e}')
                finally:
                    if tmp_dir.exists():
                        shutil.rmtree(tmp_dir)
            print(f'  {category}: {len(list(cat_dir.glob("*.jpg")))} images')

    print('Scraping images from Bing (~30 min)...')
    download_images(FRAMES_DIR, FOOD_CATEGORIES, IMAGES_PER_KEYWORD)
    print('Scraping complete')
else:
    print('Skipping scrape -- HF download succeeded')

## 6. Upload Dataset to HuggingFace

Auto-runs after scraping so teammates can skip the scrape next time.

In [ ]:
# Upload right after download (skipped automatically if no HF_TOKEN)
SKIP_DATASET_UPLOAD = False

if not SKIP_DATASET_UPLOAD and HF_TOKEN and not downloaded_from_hf:
    try:
        from huggingface_hub import HfApi, create_repo
        api = HfApi(token=HF_TOKEN)
        create_repo(HF_DATASET_REPO, repo_type='dataset', token=HF_TOKEN, exist_ok=True)
        print(f'Uploading dataset to {HF_DATASET_REPO}...')
        api.upload_folder(
            repo_id=HF_DATASET_REPO, repo_type='dataset',
            folder_path=str(FRAMES_DIR), path_in_repo='frames',
            commit_message='Upload 16-class food dataset',
            token=HF_TOKEN,
        )
        print(f'Dataset uploaded: https://huggingface.co/datasets/{HF_DATASET_REPO}')
    except Exception as e:
        print(f'Upload failed: {e}')
elif downloaded_from_hf:
    print('[SKIP] Dataset was downloaded from HF -- no need to re-upload')
elif not HF_TOKEN:
    print('[SKIP] No HF_TOKEN -- add it to Colab Secrets to enable upload')
else:
    print('[SKIP] SKIP_DATASET_UPLOAD = True')

## 7. Dataset Inspection

In [ ]:
from vit_video.data.splits import discover_videos_by_class, count_frame_images

videos_by_class = discover_videos_by_class(FRAMES_DIR)
total_frames = count_frame_images(FRAMES_DIR)

print(f'Total frames on disk: {total_frames}\n')
for cls, stems in sorted(videos_by_class.items()):
    n_frames = sum(1 for _ in (FRAMES_DIR / cls).glob('*.jpg'))
    print(f'  {cls:15s} {len(stems):3d} unique videos / {n_frames:5d} frames')

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
from vit_video.data.dataset import VideoDataset

preview_ds = VideoDataset(root=FRAMES_DIR, frames_per_video=8, img_size=224, augment=False)
print(f'Dataset: {len(preview_ds)} videos, classes: {preview_ds.classes}')

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
video_tensor, label = preview_ds[0]
for i, ax in enumerate(axes.flat):
    frame = video_tensor[i].permute(1, 2, 0).numpy()
    frame = frame * [0.229, 0.224, 0.225] + [0.485, 0.456, 0.406]
    ax.imshow(frame.clip(0, 1))
    ax.set_title(f'Frame {i}')
    ax.axis('off')
fig.suptitle(f'Sample video -- class: {preview_ds.classes[label]}', fontsize=14)
plt.tight_layout()
plt.show()

## 8. Train/Val/Test Split

In [ ]:
from vit_video.data.splits import ensure_split_manifest

manifest_path = ensure_split_manifest(
    frames_root=FRAMES_DIR,
    manifest_path=DATASET_DIR / 'video_split_manifest.json',
    train_ratio=0.7, val_ratio=0.15, test_ratio=0.15,
    seed=42,
)
print(f'Split manifest: {manifest_path}')

## 9. Training

Resumes from Drive checkpoint if available. Syncs to Drive after each best epoch.

In [ ]:
import argparse, shutil
from vit_video.train import main as train_main

MODEL_PATH = PACKAGE_ROOT / 'models' / 'best_food_classifier.pth'
MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)

# Resume from Drive if available
resume_path = ''
if DRIVE_CHECKPOINT_PATH and os.path.exists(DRIVE_CHECKPOINT_PATH):
    shutil.copy2(DRIVE_CHECKPOINT_PATH, str(MODEL_PATH))
    resume_path = str(MODEL_PATH)
    print(f'Resuming from Drive: {DRIVE_CHECKPOINT_PATH}')
else:
    print('Starting training from scratch')

train_args = argparse.Namespace(
    dataset_dir=str(FRAMES_DIR),
    epochs=20,
    batch_size=8,
    lr=3e-5,
    weight_decay=1e-3,
    max_grad_norm=1.0,
    dropout=0.4,
    num_frames=8,
    img_size=224,
    disable_augmentation=False,
    class_weighting=True,
    min_samples_per_class=20,
    temporal_pool='lstm',
    norm_mean='0.485,0.456,0.406',
    norm_std='0.229,0.224,0.225',
    hparam_search_epochs=0,
    lr_candidates='5e-6,1e-5,3e-5,5e-5,1e-4',
    num_workers=2,
    patience=7,
    min_delta=5e-5,
    backbone='auto',
    output_model=str(MODEL_PATH),
    resume_from=resume_path,
    drive_checkpoint_dir=DRIVE_CHECKPOINT_DIR,
    split_manifest=str(manifest_path),
    no_auto_split_manifest=True,
    train_ratio=0.7, val_ratio=0.15, test_ratio=0.15,
    split_seed=42,
)

train_main(train_args)

if MODEL_PATH.exists() and DRIVE_CHECKPOINT_PATH:
    shutil.copy2(str(MODEL_PATH), DRIVE_CHECKPOINT_PATH)
    print(f'Final checkpoint synced to Drive')

model_path = str(MODEL_PATH)
print(f'\nSaved model: {model_path}')

### Training History

In [ ]:
import json

history_file = MODEL_PATH.with_name(MODEL_PATH.stem + '_history.json')
if history_file.exists():
    history = json.loads(history_file.read_text())
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    ax1.plot(history['train_loss'], label='Train', marker='o', markersize=4)
    ax1.plot(history['val_loss'], label='Val', marker='s', markersize=4)
    ax1.set(xlabel='Epoch', ylabel='Loss', title='Loss')
    ax1.legend(); ax1.grid(True, alpha=0.3)
    ax2.plot(history['train_acc'], label='Train', marker='o', markersize=4)
    ax2.plot(history['val_acc'], label='Val', marker='s', markersize=4)
    ax2.set(xlabel='Epoch', ylabel='Accuracy', title='Accuracy')
    ax2.legend(); ax2.grid(True, alpha=0.3)
    plt.tight_layout(); plt.show()
else:
    print(f'No history file at {history_file}')

## 10. Evaluation on Test Split

In [ ]:
from vit_video.test import build_test_loader, evaluate, print_results, save_results
from vit_video.utils.model_utils import load_model_from_checkpoint

test_loader, classes, _ = build_test_loader(
    dataset_root=str(FRAMES_DIR), batch_size=4, num_frames=8, num_workers=2,
    split_manifest=str(manifest_path),
)
NUM_CLASSES = len(classes)
model = load_model_from_checkpoint(model_path, num_classes=NUM_CLASSES, device=device)
results = evaluate(model, test_loader, device, classes)
print_results(results)
save_results(results, PACKAGE_ROOT / 'results')

In [ ]:
import seaborn as sns
import numpy as np

cm = np.array(results['confusion_matrix'])
fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=classes, yticklabels=classes, ax=ax)
ax.set(xlabel='Predicted', ylabel='Actual', title=f'Confusion Matrix ({NUM_CLASSES} classes)')
plt.xticks(rotation=45, ha='right'); plt.yticks(rotation=0)
plt.tight_layout(); plt.show()

print(f'\nAccuracy:  {results["accuracy"]:.4f}')
print(f'Precision: {results["precision_macro"]:.4f}')
print(f'Recall:    {results["recall_macro"]:.4f}')
print(f'F1:        {results["f1_macro"]:.4f}')

## 11. Health-Label Rollup

In [ ]:
from sklearn.metrics import confusion_matrix as sk_cm, classification_report

preds = results['predictions']
truths = results['ground_truth']

health_preds = [HEALTH_LABELS.get(classes[p], '?') for p in preds]
health_truths = [HEALTH_LABELS.get(classes[t], '?') for t in truths]

health_correct = sum(p == t for p, t in zip(health_preds, health_truths))
health_acc = health_correct / len(health_preds) if health_preds else 0
print(f'Health-level accuracy: {health_acc:.2%} ({health_correct}/{len(health_preds)})')
print()

health_labels = ['healthy', 'unhealthy', 'not_food']
h_cm = sk_cm(health_truths, health_preds, labels=health_labels)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(h_cm, annot=True, fmt='d', cmap='Greens',
            xticklabels=health_labels, yticklabels=health_labels, ax=ax)
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title(f'Health-Level Confusion (acc={health_acc:.1%})')
plt.tight_layout()
plt.show()

print(classification_report(health_truths, health_preds, labels=health_labels, zero_division=0))

## 12. Inference Demo

In [ ]:
import torch
from PIL import Image
from vit_video.utils.data_utils import build_transform
import time

transform = build_transform(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
test_dirs = ['fruits', 'burgers', 'not_food']
model.eval()

fig, axes = plt.subplots(1, len(test_dirs), figsize=(5 * len(test_dirs), 5))
for ax, cls_name in zip(axes, test_dirs):
    sample = sorted(FRAMES_DIR.glob(f'{cls_name}/*.jpg'))[:1]
    if not sample:
        ax.set_title(f'{cls_name}: no images'); ax.axis('off'); continue
    img = Image.open(sample[0]).convert('RGB').resize((224, 224))
    video_tensor = torch.stack([transform(img)] * 8).unsqueeze(0).to(device)
    with torch.no_grad():
        start = time.perf_counter()
        outputs = model(video_tensor)
        ms = (time.perf_counter() - start) * 1000.0
        probs = torch.softmax(outputs, dim=1)
        idx = torch.argmax(probs, dim=1).item()
    pred_class = classes[idx]
    health = HEALTH_LABELS.get(pred_class, '?')
    ax.imshow(img)
    ax.set_title(f'{pred_class} ({probs[0][idx]:.0%})\n[{health}] {ms:.0f}ms')
    ax.axis('off')
plt.suptitle('Inference Demo', fontsize=14); plt.tight_layout(); plt.show()

## 13. Export to Mobile Formats

In [ ]:
export_script = str(PACKAGE_ROOT / 'export_mobile.py')
export_dir = PACKAGE_ROOT / 'exported_models'
results_file = PACKAGE_ROOT / 'results' / 'test_results.json'

cmd = [sys.executable, export_script,
       '--model', str(model_path), '--output-dir', str(export_dir),
       '--format', 'torchscript', 'onnx',
       '--num-classes', str(NUM_CLASSES),
       '--classes', ','.join(classes),
       '--eval-results', str(results_file)]
subprocess.run(cmd, check=False)

print('\nExported files:')
for f in sorted(export_dir.glob('*')):
    if f.is_file():
        print(f'  {f.name:40s} {f.stat().st_size / 1024 / 1024:>7.1f} MB')

## 14. Upload Model to HuggingFace

In [ ]:
# Set SKIP_MODEL_UPLOAD = False to upload trained model to HF
SKIP_MODEL_UPLOAD = False

if not SKIP_MODEL_UPLOAD and HF_TOKEN:
    HF_MODEL_REPO = 'maia2000/food-classifier'
    upload_script = str(PACKAGE_ROOT / 'upload_hf.py')
    cmd = [sys.executable, upload_script,
           '--repo-id', HF_MODEL_REPO,
           '--export-dir', str(export_dir)]
    subprocess.run(cmd, check=False)
    print(f'Model uploaded: https://huggingface.co/{HF_MODEL_REPO}')
elif not HF_TOKEN:
    print('[SKIP] No HF_TOKEN -- add it to Colab Secrets to enable upload')
else:
    print('[SKIP] Model upload')

## Model Summary

In [ ]:
total_params = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
backbone = getattr(model, 'model_name', 'auto')
temporal = getattr(model, 'temporal_pool', 'lstm')

print('Model Summary')
print(f'  Architecture:       MobileViTModel + BiLSTM')
print(f'  Backbone:           {backbone}')
print(f'  Temporal pooling:   {temporal}')
print(f'  Total parameters:   {total_params:,}')
print(f'  Trainable params:   {trainable:,}')
print(f'  Input shape:        (B, 8, 3, 224, 224)')
print(f'  Output:             (B, {NUM_CLASSES})')
print(f'  Classes:            {classes}')
print(f'  Device:             {device}')